# Run the talend_migration tests

Runs pytest from the bundle root on Databricks, so Spark unit tests use this compute's session. Set `scope` to `tests/static` for static checks only, or `tests/waves/wave_NN` for one wave. Fails if any test fails.

In [0]:
_nb_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
_root = "/Workspace" + _nb_path.rsplit("/tests/", 1)[0]

dbutils.widgets.text("root_path", _root)
dbutils.widgets.text("scope", "tests")      # tests | tests/static | tests/unit | tests/waves/wave_NN
dbutils.widgets.text("extra_args", "")      # e.g. -k notebooks  or  -m "not spark"

root = dbutils.widgets.get("root_path")
scope = dbutils.widgets.get("scope")
extra_args = dbutils.widgets.get("extra_args")

In [0]:
import os
import shlex
import sys

import pytest

sys.dont_write_bytecode = True
os.chdir(root)
# Drop cached project modules so edits to src/ and tests/ are picked up on rerun.
for name in [m for m in sys.modules if m in ("project", "operators") or m.startswith("operators.")]:
    del sys.modules[name]

exit_code = pytest.main([scope, "-q", "--color=no", "-rfEs", "-p", "no:cacheprovider", *shlex.split(extra_args)])
print("pytest exit code:", int(exit_code))

In [0]:
if int(exit_code) != 0:
    raise AssertionError(f"pytest failed with exit code {int(exit_code)} - see output above")
dbutils.notebook.exit("passed")